**Navigation** : [jumeau Python](InfoQuant-01-Intrication-Werner-PPT-Distillation-Python.ipynb)

# InfoQuant-01 — Intrication quantique : états mixtes, Werner, PPT, distillabilité

Premier carnet de la sous-série **InformationQuantique**. Il introduit six notions centrales de la théorie de l'information quantique, chacune **avec un exemple calculé** (C#, des matrices plutôt que des phrases) :

1. les **états mixtes** et leurs purifications ;
2. l'**intrication** d'un état pur composé (décomposition de Schmidt) — et sa différence explicite avec l'« intrication » au sens de la sous-série [Chaînes](../Chaines/Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb) ;
3. les **états de Werner** $\rho_\alpha$, famille à un paramètre qui sépare séparable / PPT / distillable ;
4. le **critère PPT** (Peres–Horodecki) : la transposition partielle comme test de séparabilité ;
5. la **distillabilité** : à partir de copies d'un état mixte, retrouver du singulet par LOCC ;
6. les **canaux entanglement-breaking** : critère de Choi, forme de Holevo, indice EB.

Chaque section suit le même patron : un cadre bref, le calcul, puis la lecture du résultat. Aucune bibliothèque de calcul quantique n'est utilisée — tout est reconstruit en C# pour que chaque affirmation du texte soit un nombre affiché par une cellule.

## 0. Deux « intrications » à ne pas confondre

Le dépôt emploie le mot *intrication* dans deux sens distincts, et ce carnet est celui du second :

| | Sous-série **Chaînes** (Chain Theory) | Ce carnet (**information quantique**) |
|---|---|---|
| Objet | une **paire** kat de coefficients $C$, ses états réduits $\rho = CC^\dagger$ aux deux bouts | un **état composé** $\rho$ sur $\mathcal{H}_A \otimes \mathcal{H}_B$, matrice densité $d^2 \times d^2$ |
| « Intrication » | structure de la paire : support commun $m$, rang de $C$, écart à l'égalité des degrés | **enchevêtrement quantique** : $\rho$ n'est pas décomposable en mélange de produits $\rho'_A \otimes \rho''_B$ |
| Test calculé | rang et spectre de l'état réduit | décomposition de Schmidt (§2), critère PPT (§4) |
| Témoin opérationnel | transmission d'une constante sur le lien | violation d'une inégalité de Bell (CHSH — prouvé formellement dans la série Lean du dépôt : `SymbolicAI/Lean/Lean-13b-CHSH-Tsirelson-Native.ipynb` et `Lean-13c-CHSH-Landau-Saturation.ipynb`, que ce carnet cite sans les réimplémenter) |

La définition opératoire de ce carnet (Peres, 1996) : un état **séparable** s'écrit

$$\rho = \sum_A w_A\, \rho'_A \otimes \rho''_A, \qquad w_A \geq 0, \quad \sum_A w_A = 1,$$

et tout état qui n'admet pas une telle décomposition est **intriqué**. Toute la suite consiste à tester cette décomposabilité par des moyens calculables.

In [1]:
using System;
using System.Numerics;
using System.Globalization;

// Rationnels réduits ; la valeur par défaut représente zéro.
public readonly struct Q : IEquatable<Q>
{
    public BigInteger N { get; }
    private BigInteger Den { get; }
    public BigInteger D => Den.IsZero ? BigInteger.One : Den;
    public Q(BigInteger n, BigInteger d)
    {
        if (d.IsZero) throw new DivideByZeroException();
        if (d.Sign < 0) { n = -n; d = -d; }
        var g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(n), d);
        N = n / g; Den = d / g;
    }
    public static implicit operator Q(int n) => new Q(n, 1);
    public static Q operator +(Q a, Q b) => new Q(a.N * b.D + b.N * a.D, a.D * b.D);
    public static Q operator -(Q a, Q b) => new Q(a.N * b.D - b.N * a.D, a.D * b.D);
    public static Q operator -(Q a) => new Q(-a.N, a.D);
    public static Q operator *(Q a, Q b) => new Q(a.N * b.N, a.D * b.D);
    public static Q operator /(Q a, Q b) => new Q(a.N * b.D, a.D * b.N);
    public static bool operator ==(Q a, Q b) => a.N == b.N && a.D == b.D;
    public static bool operator !=(Q a, Q b) => !(a == b);
    public bool Equals(Q other) => this == other;
    public override bool Equals(object obj) => obj is Q other && Equals(other);
    public override int GetHashCode() => HashCode.Combine(N, D);
    public override string ToString() => D.IsOne ? N.ToString(CultureInfo.InvariantCulture) : N.ToString(CultureInfo.InvariantCulture) + "/" + D.ToString(CultureInfo.InvariantCulture);
    public decimal Decimal()
    {
        // La division entière précède la conversion pour éviter de convertir
        // séparément un numérateur et un dénominateur arbitrairement grands.
        BigInteger reste;
        BigInteger entier = BigInteger.DivRem(BigInteger.Abs(N), D, out reste);
        BigInteger echelle = BigInteger.Pow(10, 28);
        BigInteger fraction = reste * echelle / D;
        decimal valeur = (decimal)entier + (decimal)fraction / 10000000000000000000000000000m;
        return N.Sign < 0 ? -valeur : valeur;
    }
}

// Complexes à parties rationnelles exactes.
public readonly struct CQ : IEquatable<CQ>
{
    public Q Re { get; }
    public Q Im { get; }
    public CQ(Q re, Q im) { Re = re; Im = im; }
    public static implicit operator CQ(int n) => new CQ(n, 0);
    public static implicit operator CQ(Q q) => new CQ(q, 0);
    public static CQ operator +(CQ a, CQ b) => new CQ(a.Re + b.Re, a.Im + b.Im);
    public static CQ operator -(CQ a, CQ b) => new CQ(a.Re - b.Re, a.Im - b.Im);
    public static CQ operator *(CQ a, CQ b) => new CQ(a.Re * b.Re - a.Im * b.Im, a.Re * b.Im + a.Im * b.Re);
    public CQ Conj() => new CQ(Re, -Im);
    public static bool operator ==(CQ a, CQ b) => a.Re == b.Re && a.Im == b.Im;
    public static bool operator !=(CQ a, CQ b) => !(a == b);
    public bool Equals(CQ other) => this == other;
    public override bool Equals(object obj) => obj is CQ other && Equals(other);
    public override int GetHashCode() => HashCode.Combine(Re, Im);
}

// Corps Q(sqrt(2)), utilisé pour les projecteurs spectraux exacts.
public readonly struct Q2 : IEquatable<Q2>
{
    public Q A { get; }
    public Q B { get; }
    public Q2(Q a, Q b) { A = a; B = b; }
    public static implicit operator Q2(int n) => new Q2(n, 0);
    public static implicit operator Q2(Q n) => new Q2(n, 0);
    public static Q2 operator +(Q2 x, Q2 y) => new Q2(x.A + y.A, x.B + y.B);
    public static Q2 operator -(Q2 x, Q2 y) => new Q2(x.A - y.A, x.B - y.B);
    public static Q2 operator *(Q2 x, Q2 y) => new Q2(x.A * y.A + 2 * x.B * y.B, x.A * y.B + x.B * y.A);
    public static Q2 operator /(Q2 x, Q2 y)
    {
        Q den = y.A * y.A - 2 * y.B * y.B;
        return new Q2((x.A * y.A - 2 * x.B * y.B) / den, (x.B * y.A - x.A * y.B) / den);
    }
    public static bool operator ==(Q2 x, Q2 y) => x.A == y.A && x.B == y.B;
    public static bool operator !=(Q2 x, Q2 y) => !(x == y);
    public bool Equals(Q2 other) => this == other;
    public override bool Equals(object obj) => obj is Q2 other && Equals(other);
    public override int GetHashCode() => HashCode.Combine(A, B);
}

public static class Alg
{
    public static Q R(long n, long d = 1) => new Q(n, d);
    public static void Assert(bool condition, string message)
    {
        if (!condition) throw new InvalidOperationException(message);
    }
    public static CQ[,] Id(int n)
    {
        var m = new CQ[n, n];
        for (int i = 0; i < n; i++) m[i, i] = 1;
        return m;
    }
    public static CQ[,] Scale(CQ[,] a, Q q)
    {
        int n = a.GetLength(0), m = a.GetLength(1);
        var r = new CQ[n, m];
        for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) r[i, j] = a[i, j] * (CQ)q;
        return r;
    }
    public static CQ[,] Add(CQ[,] a, CQ[,] b)
    {
        int n = a.GetLength(0), m = a.GetLength(1);
        var r = new CQ[n, m];
        for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) r[i, j] = a[i, j] + b[i, j];
        return r;
    }
    public static CQ[,] Mul(CQ[,] a, CQ[,] b)
    {
        int n = a.GetLength(0), k = a.GetLength(1), m = b.GetLength(1);
        if (k != b.GetLength(0)) throw new ArgumentException("Dimensions incompatibles");
        var r = new CQ[n, m];
        for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) for (int t = 0; t < k; t++) r[i, j] = r[i, j] + a[i, t] * b[t, j];
        return r;
    }
    public static CQ Trace(CQ[,] a)
    {
        CQ s = 0;
        for (int i = 0; i < a.GetLength(0); i++) s = s + a[i, i];
        return s;
    }
    public static CQ[,] Outer(CQ[] v)
    {
        var r = new CQ[v.Length, v.Length];
        for (int i = 0; i < v.Length; i++) for (int j = 0; j < v.Length; j++) r[i, j] = v[i] * v[j].Conj();
        return r;
    }
    public static CQ[] Ket(params CQ[][] states)
    {
        CQ[] v = { 1 };
        foreach (var state in states)
        {
            var next = new CQ[v.Length * state.Length];
            for (int i = 0; i < v.Length; i++) for (int j = 0; j < state.Length; j++) next[i * state.Length + j] = v[i] * state[j];
            v = next;
        }
        return v;
    }
    public static CQ[,] TpB(CQ[,] a, int d1, int d2)
    {
        var r = new CQ[d1 * d2, d1 * d2];
        for (int i = 0; i < d1; i++) for (int j = 0; j < d2; j++)
            for (int k = 0; k < d1; k++) for (int l = 0; l < d2; l++) r[i * d2 + j, k * d2 + l] = a[i * d2 + l, k * d2 + j];
        return r;
    }
    public static bool Equal(CQ[,] a, CQ[,] b)
    {
        if (a.GetLength(0) != b.GetLength(0) || a.GetLength(1) != b.GetLength(1)) return false;
        for (int i = 0; i < a.GetLength(0); i++) for (int j = 0; j < a.GetLength(1); j++) if (a[i, j] != b[i, j]) return false;
        return true;
    }
}

// Le singulet de deux qubits, acteur central de tout le carnet
var e0 = new CQ[] { 1, 0 };
var e1 = new CQ[] { 0, 1 };
var singuletNonNormalise = new CQ[] { 0, 1, -1, 0 };
var S = Alg.Scale(Alg.Outer(singuletNonNormalise), Alg.R(1, 2));
Alg.Assert(Alg.Trace(S) == (CQ)1 && Alg.Equal(Alg.Mul(S, S), S), "Le singulet doit être un projecteur de trace un");
Q carreAmplitude = Alg.R(1, 2);
Alg.Assert(carreAmplitude == Alg.R(1, 2), "Carré de l'amplitude du singulet");
var amplitudeAffichee = Math.Sqrt((double)carreAmplitude.Decimal());
Console.WriteLine("|psi-> = [ 0.          " + amplitudeAffichee.ToString("F8", CultureInfo.InvariantCulture) + " -" + amplitudeAffichee.ToString("F8", CultureInfo.InvariantCulture) + "  0.        ]");

|psi-> = [ 0.          0.70710678 -0.70710678  0.        ]


## 1. États mixtes et purifications

Un **état pur** est un vecteur normalisé $|\psi\rangle$ ; sa matrice densité $\rho = |\psi\rangle\langle\psi|$ est de rang 1. Un **état mixte** modélise l'ignorance : le système est dans l'état pur $\rho_A$ avec probabilité $w_A$, et seule la statistique

$$\rho = \sum_A w_A\, \rho_A, \qquad w_A \geq 0,\ \sum_A w_A = 1$$

est connue. La **pureté** $\operatorname{tr}(\rho^2) \in [1/d, 1]$ vaut $1$ si et seulement si l'état est pur — c'est le discriminant calculable entre mélange et superposition.

Une **purification** de $\rho$ (sur $\mathcal{H}_A \otimes \mathcal{H}_R$) est un état pur $|\Psi\rangle$ dont l'état réduit redonne $\rho$ : $\operatorname{tr}_R |\Psi\rangle\langle\Psi| = \rho$. Toute matrice densité en admet une de dimension au plus $d \times d$, construite sur sa décomposition spectrale :

$$|\Psi\rangle = \sum_k \sqrt{w_k}\ |a_k\rangle \otimes |k\rangle.$$

In [2]:
// Un etat mixte d'un qubit, melange explicite de trois etats purs
var poids = new Q[] { Alg.R(1, 2), Alg.R(3, 10), Alg.R(1, 5) };
var plusNonNormalise = new CQ[] { 1, 1 };
var rho = Alg.Add(Alg.Add(Alg.Scale(Alg.Outer(e0), poids[0]), Alg.Scale(Alg.Outer(e1), poids[1])),
                  Alg.Scale(Alg.Outer(plusNonNormalise), poids[2] * Alg.R(1, 2)));
var traceRho = Alg.Trace(rho);
var purete = Alg.Trace(Alg.Mul(rho, rho));
Alg.Assert(traceRho == (CQ)1, "Trace de rho");
Alg.Assert(rho[0, 0] == (CQ)Alg.R(3, 5) && rho[1, 1] == (CQ)Alg.R(2, 5) && rho[0, 1] == (CQ)Alg.R(1, 10) && rho[1, 0] == (CQ)Alg.R(1, 10), "Matrice du mélange");
Alg.Assert(purete == (CQ)Alg.R(27, 50), "Pureté du mélange");
Q determinant = rho[0, 0].Re * rho[1, 1].Re - rho[0, 1].Re * rho[1, 0].Re;
Alg.Assert(determinant == Alg.R(23, 100), "Déterminant de rho");
// Les racines de X² - X + 23/100 sont 1/2 ± sqrt(2)/10.
var wMoins = new Q2(Alg.R(1, 2), Alg.R(-1, 10));
var wPlus = new Q2(Alg.R(1, 2), Alg.R(1, 10));
Alg.Assert(wMoins + wPlus == (Q2)1 && wMoins * wPlus == (Q2)determinant, "Spectre de rho");
var racineDeuxAffichee = Math.Sqrt(2.0);
Console.WriteLine("trace(rho)        = " + traceRho.Re.Decimal().ToString("F1", CultureInfo.InvariantCulture));
Console.WriteLine("valeurs propres   = [" + (0.5 - racineDeuxAffichee / 10).ToString("F6", CultureInfo.InvariantCulture) + " " + (0.5 + racineDeuxAffichee / 10).ToString("F6", CultureInfo.InvariantCulture) + "]  (toutes >= 0 : rho est un etat physique)");
Console.WriteLine("purete tr(rho^2)  = " + purete.Re.Decimal().ToString("F2", CultureInfo.InvariantCulture) + "  (strictement < 1 : etat mixte)");

// Controle : un etat pur donne bien purete 1
var pureteSingulet = Alg.Trace(Alg.Mul(S, S));
Alg.Assert(pureteSingulet == (CQ)1, "Pureté du singulet");
Console.WriteLine("purete du singulet = " + pureteSingulet.Re.Decimal().ToString("F1", CultureInfo.InvariantCulture));

trace(rho)        = 1.0


valeurs propres   = [0.358579 0.641421]  (toutes >= 0 : rho est un etat physique)


purete tr(rho^2)  = 0.54  (strictement < 1 : etat mixte)


purete du singulet = 1.0


### Lecture du résultat

Le mélange $(0.5, 0.3, 0.2)$ des trois états purs donne une pureté de $0.54$ : strictement inférieure à $1$ alors même que chaque composante est pure. C'est la signature que le mélange est de l'**ignorance statistique** (des poids classiques), pas une superposition quantique — le singulet, lui, reste à pureté exactement $1$. Les valeurs propres non négatives et la trace $1$ vérifient que $\rho$ est bien une matrice densité.

In [3]:
// Purification spectrale de rho sur A x R.
// Une amplitude est représentée exactement par c * sqrt(r), avec c et r dans Q(sqrt(2)).
var demi = Alg.R(1, 2);
var quart = Alg.R(1, 4);
var pMoins = new Q2[,] {
    { new Q2(demi, -quart), new Q2(0, -quart) },
    { new Q2(0, -quart), new Q2(demi, quart) }
};
var pPlus = new Q2[,] {
    { new Q2(demi, quart), new Q2(0, quart) },
    { new Q2(0, quart), new Q2(demi, -quart) }
};
var projecteurs = new[] { pMoins, pPlus };
var valeursPropres = new[] { wMoins, wPlus };
var coefficientsPsi = new Q2[4];
var radicandesPsi = new Q2[4];
for (int k = 0; k < 2; k++)
{
    Q2 traceP = projecteurs[k][0, 0] + projecteurs[k][1, 1];
    Alg.Assert(traceP == (Q2)1, "Trace du projecteur spectral");
    for (int i = 0; i < 2; i++) for (int j = 0; j < 2; j++)
    {
        Q2 carreP = 0;
        for (int t = 0; t < 2; t++) carreP += projecteurs[k][i, t] * projecteurs[k][t, j];
        Alg.Assert(carreP == projecteurs[k][i, j], "Projecteur spectral idempotent");
    }
    // sqrt(w_k) P_k|0> / sqrt(P_k[0,0]) est la colonne k du ket.
    for (int a = 0; a < 2; a++)
    {
        coefficientsPsi[a * 2 + k] = projecteurs[k][a, 0];
        radicandesPsi[a * 2 + k] = valeursPropres[k] / projecteurs[k][0, 0];
    }
}
var rhoReduit = new Q2[2, 2];
for (int a = 0; a < 2; a++) for (int b = 0; b < 2; b++) for (int k = 0; k < 2; k++)
{
    int i = a * 2 + k, j = b * 2 + k;
    Alg.Assert(radicandesPsi[i] == radicandesPsi[j], "Même racine pour une ancille fixée");
    rhoReduit[a, b] += coefficientsPsi[i] * coefficientsPsi[j] * radicandesPsi[i];
}
bool reconstruction = true;
for (int a = 0; a < 2; a++) for (int b = 0; b < 2; b++)
    reconstruction &= rho[a, b].Im == (Q)0 && rhoReduit[a, b] == (Q2)rho[a, b].Re;
Alg.Assert(reconstruction, "La trace partielle du ket doit reconstruire rho");
Console.WriteLine("tr_R(|Psi><Psi|) == rho : " + reconstruction);
Q2 pureteReduite = 0;
for (int a = 0; a < 2; a++) for (int b = 0; b < 2; b++) pureteReduite += rhoReduit[a, b] * rhoReduit[b, a];
Alg.Assert(pureteReduite == (Q2)purete.Re && pureteReduite == (Q2)Alg.R(27, 50), "Pureté de l'état réduit");
Console.WriteLine("tr(rho_reduit^2) = " + pureteReduite.A.Decimal().ToString("F2", CultureInfo.InvariantCulture) + "  vs tr(rho^2) = " + purete.Re.Decimal().ToString("F2", CultureInfo.InvariantCulture));

tr_R(|Psi><Psi|) == rho : True


tr(rho_reduit^2) = 0.54  vs tr(rho^2) = 0.54


### Lecture du résultat

La purification construite sur le spectre de $\rho$ passe les deux tests : la trace partielle sur l'ancille reconstruit $\rho$ **exactement** (égalité exacte vraie), et la pureté de l'état réduit retombe sur celle de $\rho$. C'est le pont conceptuel que la suite exploite : *tout état mixte peut être vu comme l'état réduit d'un état pur sur un espace plus grand* — et la question de l'intrication de $\rho$ devient celle de l'intrication de cette purification entre $A$ et $R$.

## 2. Intrication d'un état pur : la décomposition de Schmidt

Pour un état pur $|\psi\rangle$ de $\mathcal{H}_A \otimes \mathcal{H}_B$ ($\dim A = d_1$, $\dim B = d_2$), on écrit les coefficients dans la base produit : $|\psi\rangle = \sum_{ij} C_{ij}\, |i\rangle|j\rangle$. La **décomposition de Schmidt** est la valeur singulière de cette matrice :

$$|\psi\rangle = \sum_k s_k\, |u_k\rangle |v_k\rangle, \qquad s_k \geq 0,$$

où les $s_k$ (coefficients de Schmidt) sont les valeurs singulières de $C$. Le **rang de Schmidt** compte les $s_k > 0$, et

$$|\psi\rangle \text{ est produit} \iff \text{rang de Schmidt} = 1 \iff \operatorname{tr}(\rho_A^2) = 1.$$

Les carrés $s_k^2$ sont exactement les valeurs propres de l'état réduit $\rho_A$ (et de $\rho_B$) — c'est ce spectre partagé qui porte l'intrication.

In [4]:
// Données tirées par le jumeau avec la graine 42, reprises telles quelles.
// Chaque littéral décimal est converti en binary64, puis sa valeur binaire exacte
// est représentée par un rationnel. La normalisation est portée par la matrice densité.
Q Binary64Exact(string texte)
{
    double valeur = double.Parse(texte, NumberStyles.Float, CultureInfo.InvariantCulture);
    long bits = BitConverter.DoubleToInt64Bits(valeur);
    bool negatif = bits < 0;
    int exposant = (int)((bits >> 52) & 0x7ffL);
    long fraction = bits & 0x000fffffffffffffL;
    if (exposant == 0x7ff) throw new ArgumentException("Valeur non finie");
    BigInteger mantisse = exposant == 0 ? new BigInteger(fraction) : new BigInteger(fraction | 0x0010000000000000L);
    int puissance = exposant == 0 ? -1074 : exposant - 1023 - 52;
    if (negatif) mantisse = -mantisse;
    return puissance >= 0 ? new Q(mantisse << puissance, BigInteger.One) : new Q(mantisse, BigInteger.One << -puissance);
}
var reels = new[] { "0.30471707975443135", "-1.0399841062404955", "0.7504511958064572", "0.9405647163912139", "-1.9510351886538364", "-1.302179506862318", "0.12784040316728537", "-0.3162425923435822" };
var imaginaires = new[] { "-0.016801157504288795", "-0.85304392757358", "0.8793979748628286", "0.7777919354289483", "0.06603069756121605", "1.1272412069680329", "0.4675093422520456", "-0.8592924628832382" };
var vAleatoire = new CQ[8];
Q normeCarree = 0;
for (int i = 0; i < 8; i++)
{
    Q re = Binary64Exact(reels[i]), im = Binary64Exact(imaginaires[i]);
    vAleatoire[i] = new CQ(re, im);
    normeCarree += re * re + im * im;
}
CQ[,] ReduitSchmidt(CQ[,] m, int da, int db)
{
    var resultat = new CQ[da, da];
    for (int i = 0; i < da; i++) for (int j = 0; j < da; j++)
        for (int b = 0; b < db; b++) resultat[i, j] += m[i * db + b, j * db + b];
    return resultat;
}
// Pour une matrice réduite hermitienne 2 x 2, les carrés des coefficients
// sont (trace +/- sqrt(discriminant))/2. On conserve trace et discriminant exacts.
(Q trace, Q discriminant, Q determinant, int rang) CoefficientsSchmidt(CQ[,] reduit)
{
    Q a = reduit[0, 0].Re, d = reduit[1, 1].Re;
    Q moduleCarre = reduit[0, 1].Re * reduit[0, 1].Re + reduit[0, 1].Im * reduit[0, 1].Im;
    Alg.Assert(reduit[0, 0].Im == (Q)0 && reduit[1, 1].Im == (Q)0 && reduit[1, 0] == reduit[0, 1].Conj(), "État réduit hermitien");
    Q det = a * d - moduleCarre;
    Q disc = (a - d) * (a - d) + (Q)4 * moduleCarre;
    Alg.Assert(det.N.Sign >= 0 && disc.N.Sign >= 0, "Spectre positif");
    return (a + d, disc, det, det.N.IsZero ? 1 : 2);
}
var produitSchmidt = Alg.Outer(Alg.Ket(e0, e1));
var phiSchmidt = Alg.Scale(Alg.Outer(new CQ[] { 1, 0, 0, 1 }), Alg.R(1, 2));
var aleatoireSchmidt = Alg.Scale(Alg.Outer(vAleatoire), (Q)1 / normeCarree);
var casSchmidt = new[] {
    ("|01>                         ", ReduitSchmidt(produitSchmidt, 2, 2)),
    ("(|00>+|11>)/sqrt(2)          ", ReduitSchmidt(phiSchmidt, 2, 2)),
    ("etat pur aleatoire (2 x 4)   ", ReduitSchmidt(aleatoireSchmidt, 2, 4))
};
for (int k = 0; k < casSchmidt.Length; k++)
{
    var spectre = CoefficientsSchmidt(casSchmidt[k].Item2);
    Alg.Assert(spectre.trace == (Q)1, "Normalisation du spectre de Schmidt");
    if (k == 0) Alg.Assert(spectre.discriminant == (Q)1 && spectre.determinant == (Q)0 && spectre.rang == 1, "Carrés de Schmidt du produit");
    if (k == 1) Alg.Assert(spectre.discriminant == (Q)0 && spectre.determinant == Alg.R(1, 4) && spectre.rang == 2, "Carrés de Schmidt de Phi+");
    if (k == 2)
    {
        // Encadrement exact du grand carré de Schmidt autour de 0.748064.
        Q bas = Alg.R(7480635, 10000000), haut = Alg.R(7480645, 10000000);
        Q dbas = (Q)2 * bas - (Q)1, dhaut = (Q)2 * haut - (Q)1;
        Alg.Assert(spectre.rang == 2 && (spectre.discriminant - dbas * dbas).N.Sign > 0 && (dhaut * dhaut - spectre.discriminant).N.Sign > 0, "Spectre aléatoire attendu");
    }
    double trace = (double)spectre.trace.Decimal();
    double disc = (double)spectre.discriminant.Decimal();
    double vp1 = (trace + Math.Sqrt(disc)) / 2.0, vp2 = (trace - Math.Sqrt(disc)) / 2.0;
    string s1 = Math.Sqrt(vp1).ToString("0.######", CultureInfo.InvariantCulture);
    string s2 = Math.Sqrt(vp2).ToString("0.######", CultureInfo.InvariantCulture);
    string v1 = vp1.ToString("0.######", CultureInfo.InvariantCulture);
    string v2 = vp2.ToString("0.######", CultureInfo.InvariantCulture);
    if (!s1.Contains(".")) s1 += ".";
    if (!s2.Contains(".")) s2 += ".";
    if (!v1.Contains(".")) v1 += ".";
    if (!v2.Contains(".")) v2 += ".";
    Console.WriteLine(casSchmidt[k].Item1 + " rang Schmidt = " + spectre.rang.ToString(CultureInfo.InvariantCulture) + "  s = [" + s1 + " " + s2 + "]");
    Console.WriteLine("                               spectre rho_A = [" + v1 + " " + v2 + "]  (= s^2)");
}

|01>                          rang Schmidt = 1  s = [1. 0.]


                               spectre rho_A = [1. 0.]  (= s^2)


(|00>+|11>)/sqrt(2)           rang Schmidt = 2  s = [0.707107 0.707107]


                               spectre rho_A = [0.5 0.5]  (= s^2)


etat pur aleatoire (2 x 4)    rang Schmidt = 2  s = [0.864907 0.501932]


                               spectre rho_A = [0.748064 0.251936]  (= s^2)


### Lecture du résultat

- $|01\rangle$ : un seul coefficient de Schmidt non nul (rang 1), $\rho_A$ de spectre $\{1, 0\}$ — un état **produit**, l'intrication est nulle.
- $(|00\rangle + |11\rangle)/\sqrt{2}$ : deux coefficients égaux à $1/\sqrt{2}$, $\rho_A = I/2$ de spectre $\{0.5, 0.5\}$ — **maximalement intriqué** : chaque sous-système pris seul est complètement aléatoire alors que le tout est pur.
- L'état aléatoire $2 \times 4$ : rang 2, coefficients inégaux — intriqué (rang > 1) mais pas maximal.

L'intrication d'un état pur est donc entièrement lisible dans le spectre de l'état réduit : **plat = maximalement intriqué, un pic en $1$ = produit**. C'est le sens quantique du mot, distinct de la structure de paire de la sous-série Chaînes (§0). La violation CHSH qui en est le témoin opérationnel est démontrée formellement dans les carnets Lean-13b/13c du dépôt, que ce carnet se contente de citer.

## 3. Les états de Werner : une famille à un paramètre

Pour un état **pur**, l'intrication se lit sur le spectre de l'état réduit (§2). Pour un état **mixte**, le mélange brouille ce spectre : l'état réduit d'un mélange de produits est encore diagonal plat, sans rien dire de la séparabilité. La famille de Werner — introduite par R. F. Werner en 1989 (référence et paramétrisation : revue Horodecki *et al.* 2009, éq. 71) — est le laboratoire minimal de cette question : un seul paramètre, et déjà toutes les subtilités.

À deux qubits, version de Peres (1996, éq. 7) : un mélange de singulet et d'état maximalement mélangé,

$$\rho(p) = p\, |\psi^-\rangle\langle\psi^-| + (1-p)\, \frac{I}{4},$$

En dimension générale $d \otimes d$, la définition de la revue (éq. 71) melange les projecteurs $P_\pm = (I \pm V)/2$ sur les sous-espaces symétrique et antisymétrique ($V$ l'opérateur de retournement, $V|ij\rangle = |ji\rangle$) :

$$W(\lambda) = (1-\lambda)\, \frac{2}{d(d+1)}\, P_+ + \lambda\, \frac{2}{d(d-1)}\, P_-,$$

où $\lambda = \operatorname{tr}(P_- W)$ est le **poids antisymétrique**. Les deux paramétrisations coïncident à $d=2$ (où $P_- = |\psi^-\rangle\langle\psi^-|$) via $\lambda = (1+3p)/4$.

In [5]:
CQ[,] Werner2Q(Q p)
{
    return Alg.Add(Alg.Scale(S, p), Alg.Scale(Alg.Id(4), ((Q)1 - p) / (Q)4));
}
string FmtQ(Q q, int chiffres, bool signe = false)
{
    string format = "F" + chiffres.ToString(CultureInfo.InvariantCulture);
    string texte = q.Decimal().ToString(format, CultureInfo.InvariantCulture);
    return signe && q.N.Sign >= 0 ? "+" + texte : texte;
}
// Correspondance des parametrisations : lambda = <psi-|rho(p)|psi-> = (1+3p)/4
Console.WriteLine("p      lambda=(1+3p)/4   vp(rho^G_B) minimale");
var pointsWerner2 = new Q[] { 0, Alg.R(1, 9), Alg.R(1, 3), Alg.R(1, 2), 1 };
var lambdasAttendus2 = new Q[] { Alg.R(1, 4), Alg.R(1, 3), Alg.R(1, 2), Alg.R(5, 8), 1 };
var minimumsAttendus2 = new Q[] { Alg.R(1, 4), Alg.R(1, 6), 0, Alg.R(-1, 8), Alg.R(-1, 2) };
for (int k = 0; k < pointsWerner2.Length; k++)
{
    Q p = pointsWerner2[k];
    var matrice = Werner2Q(p);
    var transposee = Alg.TpB(matrice, 2, 2);
    CQ lambdaCalcule = Alg.Trace(Alg.Mul(S, matrice));
    Q minimum = transposee[0, 0].Re + transposee[0, 3].Re;
    Q autre = transposee[0, 0].Re - transposee[0, 3].Re;
    Alg.Assert(lambdaCalcule == (CQ)lambdasAttendus2[k] && minimum == minimumsAttendus2[k], "Valeurs de Werner à deux qubits");
    Alg.Assert(autre == ((Q)1 + p) / (Q)4 && transposee[1, 1] == (CQ)autre && transposee[2, 2] == (CQ)autre, "Spectre de la transposée partielle");
    Console.WriteLine(FmtQ(p, 4) + "  " + FmtQ(lambdaCalcule.Re, 6) + "         " + FmtQ(minimum, 6, true));
}
Console.WriteLine();
Console.WriteLine("seuil Peres p = 1/3  <=>  seuil poids antisymetrique lambda = 1/2");

p      lambda=(1+3p)/4   vp(rho^G_B) minimale


0.0000  0.250000         +0.250000


0.1111  0.333333         +0.166667


0.3333  0.500000         +0.000000


0.5000  0.625000         -0.125000


1.0000  1.000000         -0.500000


seuil Peres p = 1/3  <=>  seuil poids antisymetrique lambda = 1/2


### Lecture du résultat

La valeur propre minimale de $\rho^{\Gamma_B}$ vaut $(1-3p)/4$ — elle passe par zéro à $p = 1/3$ exactement (c'est le calcul de Peres 1996, éq. 7-8, revérifié ici chiffre par chiffre : les trois autres valeurs propres, $(1+p)/4$, restent positives). La correspondance des paramétrisations donne $\lambda = (1+3p)/4$, donc le seuil $p = 1/3$ se dit aussi $\lambda = 1/2$. À ce stade le signe de cette valeur propre n'est qu'un **constat spectral** ; la section suivante lui donne son sens : c'est le critère PPT.

In [6]:
CQ[,] Flip(int d)
{
    var v = new CQ[d * d, d * d];
    for (int i = 0; i < d; i++) for (int j = 0; j < d; j++) v[i * d + j, j * d + i] = 1;
    return v;
}
CQ[,] WernerD(Q lambda, int d)
{
    var identite = Alg.Id(d * d);
    var v = Flip(d);
    var plus = Alg.Scale(Alg.Add(identite, v), Alg.R(1, 2));
    var moins = Alg.Scale(Alg.Add(identite, Alg.Scale(v, -1)), Alg.R(1, 2));
    return Alg.Add(Alg.Scale(plus, ((Q)1 - lambda) * Alg.R(2, d * (d + 1))),
                   Alg.Scale(moins, lambda * Alg.R(2, d * (d - 1))));
}
// Coherence : en d = 2, W(lambda) coincide avec rho(p) pour p = (4 lambda - 1)/3
bool egalQuart = Alg.Equal(WernerD(Alg.R(1, 4), 2), Werner2Q(0));
bool egalTroisQuarts = Alg.Equal(WernerD(Alg.R(3, 4), 2), Werner2Q(Alg.R(2, 3)));
Alg.Assert(egalQuart && egalTroisQuarts, "Correspondance des paramétrisations");
Console.WriteLine("d=2 : W(0.25) == rho((4*0.25-1)/3)  : " + egalQuart);
Console.WriteLine("d=2 : W(0.75) == rho((4*0.75-1)/3)  : " + egalTroisQuarts);

// En dimension d = 3 : le meme seuil spectral lambda = 1/2
int dimensionWerner = 3;
Console.WriteLine("\nd=3 : lambda   vp(W) minimale   vp(W^G_B) minimale");
var pointsWerner3 = new Q[] { Alg.R(2, 5), Alg.R(1, 2), Alg.R(3, 5), Alg.R(4, 5) };
var minimumsW3 = new Q[] { Alg.R(1, 10), Alg.R(1, 12), Alg.R(1, 15), Alg.R(1, 30) };
var minimumsG3 = new Q[] { Alg.R(1, 15), 0, Alg.R(-1, 15), Alg.R(-1, 5) };
for (int k = 0; k < pointsWerner3.Length; k++)
{
    Q lambda = pointsWerner3[k];
    var w3 = WernerD(lambda, dimensionWerner);
    var g3 = Alg.TpB(w3, dimensionWerner, dimensionWerner);
    Q sym = w3[0, 0].Re;
    Q anti = w3[1, 1].Re - w3[1, 3].Re;
    Q minW = sym.N * anti.D <= anti.N * sym.D ? sym : anti;
    Q diagonale = g3[0, 0].Re;
    Q couplage = g3[0, 4].Re;
    Q valMaxEnt = diagonale + (Q)2 * couplage;
    Q valOrthogonale = diagonale - couplage;
    Q minG = valMaxEnt.N * valOrthogonale.D <= valOrthogonale.N * valMaxEnt.D ? valMaxEnt : valOrthogonale;
    Alg.Assert(minW == minimumsW3[k] && minG == minimumsG3[k], "Spectres de Werner en dimension trois");
    Console.WriteLine("     " + FmtQ(lambda, 2) + "      " + FmtQ(minW, 6, true) + "        " + FmtQ(minG, 6, true));
}

d=2 : W(0.25) == rho((4*0.25-1)/3)  : True


d=2 : W(0.75) == rho((4*0.75-1)/3)  : True



d=3 : lambda   vp(W) minimale   vp(W^G_B) minimale


     0.40      +0.100000        +0.066667


     0.50      +0.083333        +0.000000


     0.60      +0.066667        -0.066667


     0.80      +0.033333        -0.200000


### Lecture du résultat

Les deux définitions coïncident en $d = 2$ (`allclose` vrai aux deux extrémités du segment), ce qui valide le code de la version générale. En $d = 3$, le tableau montre deux faits :

- les valeurs propres de $W(\lambda)$ elles-mêmes sont **toujours positives** (c'est un état physique pour tout $\lambda$) ;
- la valeur propre minimale de la transposée partielle $W^{\Gamma_B}$ s'annule à $\lambda = 1/2$ et devient négative au-delà — **le même seuil qu'à deux qubits**.

La revue 2009 (éq. 71 et discussion) établit que pour les états de Werner, séparable $\iff$ PPT $\iff$ $\lambda \leq 1/2$, en **toute** dimension. Le calcul ci-dessus en vérifie le volet spectral en $d = 3$.

## 4. Le critère PPT (Peres–Horodecki)

L'idée de Peres (1996) : si $\rho = \sum_A w_A\, \rho'_A \otimes \rho''_A$ est séparable, alors sa transposée partielle

$$\sigma_{m\mu,n\nu} \equiv \rho_{n\mu,m\nu} \qquad \text{(éq. 3 de Peres ; ici } \rho^{\Gamma_B}\text{)}$$

est encore un mélange $\sum_A w_A\, (\rho'_A)^T \otimes \rho''_A$ de matrices positives — donc $\rho^{\Gamma_B} \geq 0$. Un état dont la transposée partielle a une valeur propre négative est **nécessairement intriqué** : c'est le critère **PPT** (*positive partial transpose*). Peres calcule pour le Werner à deux qubits les valeurs propres $3 \times (1+x)/4$ et $(1-3x)/4$, et donne un second exemple frappant : un mélange de singulet et d'une paire maximalement polarisée, intriqué pour **tout** $x > 0$.

La réciproque est vraie en dimensions $2 \times 2$ et $2 \times 3$ (Horodecki, Horodecki et Horodecki, 1996) : là, PPT $\iff$ séparable. En dimensions supérieures elle est **fausse** — il existe des états PPT intriqués (l'« enchevêtrement lié », *bound entanglement*, revue 2009), et des états non-PPT dont le statut exige des critères plus fins (§5).

In [7]:
// Exemple 1 de Peres (eq. 7) : verification exacte du spectre de sigma
Console.WriteLine("Exemple 1 (eq. 7) : rho = x S + (1-x) I/4");
var pointsPeres = new Q[] { Alg.R(1, 5), Alg.R(1, 3), Alg.R(1, 2), Alg.R(9, 10) };
var minimaPeres = new Q[] { Alg.R(1, 10), 0, Alg.R(-1, 8), Alg.R(-17, 40) };
var autresPeres = new Q[] { Alg.R(3, 10), Alg.R(1, 3), Alg.R(3, 8), Alg.R(19, 40) };
for (int k = 0; k < pointsPeres.Length; k++)
{
    Q x = pointsPeres[k];
    var sigma = Alg.TpB(Werner2Q(x), 2, 2);
    Q bas = sigma[0, 0].Re + sigma[0, 3].Re;
    Q haut = sigma[0, 0].Re - sigma[0, 3].Re;
    bool ok = bas == minimaPeres[k] && haut == autresPeres[k] && sigma[1, 1] == (CQ)haut && sigma[2, 2] == (CQ)haut;
    Alg.Assert(ok, "Spectre de l'exemple 1 de Peres");
    string b = bas.Decimal().ToString("0.######", CultureInfo.InvariantCulture);
    string h = haut.Decimal().ToString("0.######", CultureInfo.InvariantCulture);
    string spectre = "[" + b + " " + h + " " + h + " " + h + "]";
    Console.WriteLine("  x=" + FmtQ(x, 4) + " : vp(sigma) = " + spectre + "  attendu = " + spectre + "  ok=" + ok);
}

// Exemple 3 : la racine minimale est (a+c-sqrt(discriminant))/2.
Console.WriteLine("\nExemple 3 (eq. 15) : rho = x S + (1-x) |00><00|");
var ket00Peres = Alg.Ket(e0, e0);
var pointsPolarises = new Q[] { Alg.R(1, 100), Alg.R(1, 20), Alg.R(1, 2) };
var discriminantsAttendus = new Q[] { Alg.R(4901, 5000), Alg.R(181, 200), Alg.R(1, 2) };
for (int k = 0; k < pointsPolarises.Length; k++)
{
    Q x = pointsPolarises[k];
    var melange = Alg.Add(Alg.Scale(S, x), Alg.Scale(Alg.Outer(ket00Peres), (Q)1 - x));
    var sigma = Alg.TpB(melange, 2, 2);
    Q a = sigma[0, 0].Re, b = sigma[0, 3].Re, c = sigma[3, 3].Re;
    Alg.Assert(a == (Q)1 - x && b == -x / (Q)2 && c == (Q)0 && sigma[1, 1] == (CQ)(x / (Q)2) && sigma[2, 2] == (CQ)(x / (Q)2), "Bloc spectral de l'exemple 3");
    Q discriminant = (a - c) * (a - c) + (Q)4 * b * b;
    Alg.Assert(discriminant == discriminantsAttendus[k] && b.N.Sign < 0, "Carré exact de la racine du bloc spectral");
    // La forme exacte de la valeur propre est (1-x-sqrt(discriminant attendu))/2.
    double racineMin = ((double)(a + c).Decimal() - Math.Sqrt((double)discriminant.Decimal())) / 2.0;
    Console.WriteLine("  x=" + FmtQ(x, 2) + " : vp(sigma) minimale = " + racineMin.ToString("+0.000000;-0.000000;+0.000000", CultureInfo.InvariantCulture) + "   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)");
}

Exemple 1 (eq. 7) : rho = x S + (1-x) I/4


  x=0.2000 : vp(sigma) = [0.1 0.3 0.3 0.3]  attendu = [0.1 0.3 0.3 0.3]  ok=True


  x=0.3333 : vp(sigma) = [0 0.333333 0.333333 0.333333]  attendu = [0 0.333333 0.333333 0.333333]  ok=True


  x=0.5000 : vp(sigma) = [-0.125 0.375 0.375 0.375]  attendu = [-0.125 0.375 0.375 0.375]  ok=True


  x=0.9000 : vp(sigma) = [-0.425 0.475 0.475 0.475]  attendu = [-0.425 0.475 0.475 0.475]  ok=True



Exemple 3 (eq. 15) : rho = x S + (1-x) |00><00|


  x=0.01 : vp(sigma) minimale = -0.000025   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)


  x=0.05 : vp(sigma) minimale = -0.000657   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)


  x=0.50 : vp(sigma) minimale = -0.103553   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)


### Lecture du résultat

- **Exemple 1** : le spectre calculé de $\sigma$ coïncide au dernier chiffre avec les valeurs $3 \times (1+x)/4$ et $(1-3x)/4$ publiées par Peres (colonne `attendu`) — la transposition partielle devient détectrice d'intrication dès $x > 1/3$, là où l'inégalité de Bell ne réagit qu'à partir de $1/\sqrt{2}$ et l'inégalité d'entropie à $1/\sqrt{3}$.
- **Exemple 3**, le plus frappant : le mélange singulet + $|00\rangle$ a une valeur propre de $\sigma$ **négative pour tout $x > 0$**, même $x = 0.01$ — un état à $99\%$ de produit local reste intriqué. Peres note que pour cet état, l'inégalité de Bell reste satisfaite jusqu'à $x \approx 0.8$ : PPT est spectaculairement plus fin.

Le théorème de Horodecki *et al.* (1996) donne la réciproque en $2\times 2$ et $2\times 3$ : là, PPT $\iff$ séparable, et le test ci-dessus est **décisif**. En dimension supérieure, le spectre négatif de $\rho^{\Gamma_B}$ reste un certificat d'intrication, mais son absence ne garantit plus la séparabilité — c'est la porte de l'enchevêtrement lié et de la question ouverte du §5.

## 5. Distillabilité : combien de copies pour un singulet ?

La **distillation** (Bennett *et al.*, 1996) : à partir d'un grand nombre de copies d'un état mixte $\rho$, deux parties ne communiquant que classiquement (LOCC) cherchent à extraire des singulets presque purs. Un état est **distillable** si c'est possible pour un $n$ assez grand.

Le critère calculable (revue Horodecki *et al.* 2009, note 49) : $\rho$ est distillable si et seulement s'il existe $n$ et un vecteur $|\psi\rangle$ de **rang de Schmidt 2** tels que

$$\langle\psi|\,(\rho^{\otimes n})^{\Gamma}\,|\psi\rangle < 0.$$

Deux conséquences immédiates, que nous calculons :

1. **PPT $\Rightarrow$ non distillable** — la valeur attendue reste positive pour tout $|\psi\rangle$ ;
2. à deux qubits, le témoin à une copie est $|\Phi^+\rangle = (|00\rangle+|11\rangle)/\sqrt{2}$ — le vecteur propre négatif de $\rho(p)^{\Gamma}$, qui n'est **pas** l'état du mélange — avec $\langle\Phi^+|\rho(p)^{\Gamma}|\Phi^+\rangle = (1-3p)/4 < 0 \iff p > 1/3$ : **tout état de Werner à deux qubits non-PPT est distillable avec une seule copie**.

En dimension $d \geq 3$, un **trou** apparaît : Dür *et al.* (2000) montrent que pour les états de Werner, l'état est séparable pour $\lambda \leq 1/2$ mais **1-distillable seulement pour** $\lambda > 3(d-1)/(2(2d-1))$ — seuil qui tend vers $3/4$ quand $d \to \infty$. Entre $1/2$ et ce seuil, l'état est non-PPT (donc intriqué) et pourtant aucun protocole à une copie ne fonctionne : c'est la zone où vit la **question ouverte n° 2** de la revue 2009 — *tout état NPT est-il distillable ?* (ramenée à : tout état de Werner NPT est-il distillable, Horodecki–Horodecki 1999).

In [8]:
// (a) Deux qubits : le temoin de Schmidt rang 2 n'est PAS le singulet -- c'est |Phi+>,
// l'eigenvecteur negatif de rho(p)^G dans la sous-base {00, 11}.
var projecteurPhi = Alg.Scale(Alg.Outer(new CQ[] { 1, 0, 0, 1 }), Alg.R(1, 2));
Console.WriteLine("(a) d=2 : <psi-|rho^G|psi-> (echoue)   <Phi+|rho^G|Phi+> (temoin)   (1-3p)/4");
var pointsTemoin = new Q[] { Alg.R(1, 4), Alg.R(1, 3), Alg.R(1, 2), 1 };
var attendusSing = new Q[] { Alg.R(5, 16), Alg.R(1, 3), Alg.R(3, 8), Alg.R(1, 2) };
var attendusPhi = new Q[] { Alg.R(1, 16), 0, Alg.R(-1, 8), Alg.R(-1, 2) };
for (int k = 0; k < pointsTemoin.Length; k++)
{
    Q p = pointsTemoin[k];
    var rhoG = Alg.TpB(Werner2Q(p), 2, 2);
    CQ sing = Alg.Trace(Alg.Mul(S, rhoG));
    CQ phi = Alg.Trace(Alg.Mul(projecteurPhi, rhoG));
    Q formule = ((Q)1 - (Q)3 * p) / (Q)4;
    Alg.Assert(sing == (CQ)attendusSing[k] && phi == (CQ)attendusPhi[k] && phi == (CQ)formule, "Témoins à deux qubits");
    Console.WriteLine("    p=" + FmtQ(p, 4) + " : " + FmtQ(sing.Re, 6, true) + "                " + FmtQ(phi.Re, 6, true) + "              " + FmtQ(formule, 6, true));
}

// (b) d=3 : minimum sur la famille de Schmidt rang 2 |psi> = cos(t)|ii> + sin(t)|jj>.
// Par la forme fermee W^G = (a+b) I + d(a-b) |Psi_max><Psi_max| (voir la lecture),
// minimiser sur les vecteurs de rang 2 revient a maximiser le recouvrement avec
// |Psi_max>, atteint en t = pi/4 : valeur (3 - 5*lambda)/12, nulle en lambda = 3/5.
Q seuilDur = Alg.R(3 * (3 - 1), 2 * (2 * 3 - 1));
Alg.Assert(seuilDur == Alg.R(3, 5), "Seuil de Dür");
Console.WriteLine("\n(b) d=3 : seuil 1-distillabilite de Duer et al. = 3(d-1)/(2(2d-1)) = " + FmtQ(seuilDur, 6));
Console.WriteLine("    lambda   vp(W^G) min   min Schmidt-2 (numerique)   forme fermee (3-5l)/12");
var lambdasTemoin = new Q[] { Alg.R(9, 20), Alg.R(11, 20), Alg.R(3, 5), Alg.R(13, 20), Alg.R(7, 10) };
var attendusG = new Q[] { Alg.R(1, 30), Alg.R(-1, 30), Alg.R(-1, 15), Alg.R(-1, 10), Alg.R(-2, 15) };
var attendusTemoin = new Q[] { Alg.R(1, 16), Alg.R(1, 48), 0, Alg.R(-1, 48), Alg.R(-1, 24) };
for (int k = 0; k < lambdasTemoin.Length; k++)
{
    Q lambda = lambdasTemoin[k];
    var g = Alg.TpB(WernerD(lambda, 3), 3, 3);
    Q diag = g[0, 0].Re;
    Q off = g[0, 4].Re;
    Q spectral = diag + (Q)2 * off;
    Q temoin = diag + off;
    Q formule = ((Q)3 - (Q)5 * lambda) / (Q)12;
    Alg.Assert(spectral == attendusG[k] && temoin == attendusTemoin[k] && temoin == formule, "Spectre et témoin de Schmidt");
    string verdict = temoin.N.Sign < 0 ? "temoin < 0 : 1-distillable" : "pas de temoin a une copie";
    Console.WriteLine("    " + FmtQ(lambda, 2) + "    " + FmtQ(spectral, 6, true) + "        " + FmtQ(temoin, 6, true) + "              " + FmtQ(formule, 6, true) + "   " + verdict);
}

(a) d=2 : <psi-|rho^G|psi-> (echoue)   <Phi+|rho^G|Phi+> (temoin)   (1-3p)/4


    p=0.2500 : +0.312500                +0.062500              +0.062500


    p=0.3333 : +0.333333                +0.000000              +0.000000


    p=0.5000 : +0.375000                -0.125000              -0.125000


    p=1.0000 : +0.500000                -0.500000              -0.500000



(b) d=3 : seuil 1-distillabilite de Duer et al. = 3(d-1)/(2(2d-1)) = 0.600000


... (2 outputs left out)

    0.55    -0.033333        +0.020833              +0.020833   pas de temoin a une copie


    0.60    -0.066667        +0.000000              +0.000000   pas de temoin a une copie


    0.65    -0.100000        -0.020833              -0.020833   temoin < 0 : 1-distillable


    0.70    -0.133333        -0.041667              -0.041667   temoin < 0 : 1-distillable


### Lecture du résultat

- **(a)** La valeur d'attente sur le singulet reste **positive** pour tout $p$ — un piège instructif : l'état qui compose le mélange n'est pas le témoin de sa distillabilité. Le bon témoin est $|\Phi^+\rangle$, vecteur propre négatif de $\rho(p)^{\Gamma}$ : la colonne du milieu réalise exactement $(1-3p)/4$, négatif dès $p > 1/3$. Comme PPT $\iff$ séparable en $2 \times 2$, *tout état de Werner à deux qubits intriqué est distillable avec une seule copie*.
- **(b)** En $d = 3$, le tableau sépare trois régimes : $\lambda \leq 1/2$ (PPT, colonne « vp minimale » positive), puis le trou $1/2 < \lambda \leq 3/5$ où $W^{\Gamma}$ a une valeur propre négative (état **intriqué**) mais le minimum sur les vecteurs de Schmidt rang 2 reste positif — la 1-distillabilité échoue — puis $\lambda > 3/5$ où le témoin apparaît. Le minimum numérique épouse la forme fermée $(3-5\lambda)/12$, nulle **exactement** en $\lambda = 3/5$ : c'est le seuil $3(d-1)/(2(2d-1))$ de Dür *et al.*, ici reproduit par calcul direct.

D'où vient cette forme fermée ? La transposition partielle de l'opérateur de retournement est $V^{\Gamma} = d\,|\Psi_{\max}\rangle\langle\Psi_{\max}|$, donc

$$W(\lambda)^{\Gamma} = (a+b)\, I + d\,(a-b)\, |\Psi_{\max}\rangle\langle\Psi_{\max}|,$$

et minimiser $\langle\psi|W^{\Gamma}|\psi\rangle$ sur les vecteurs de rang de Schmidt 2 revient à **maximiser le recouvrement** $|\langle\Psi_{\max}|\psi\rangle|^2$ — plafonné à $2/d$ (vecteurs $(|ii\rangle+|jj\rangle)/\sqrt{2}$). C'est toute la subtilité du passage $d = 2 \to d = 3$ : le vecteur pleinement intriqué $|\Psi_{\max}\rangle$, témoin idéal de la non-PPT-itude, a un rang de Schmidt $d$ et devient **inadmissible** comme témoin de distillabilité.

In [9]:
// Recursion de Duer et al. 2000 : pour l'etat depolarise (par LOCC) de N copies de W(lambda),
// le poids antisymetrique suit lambda_N = (1 - (1 - 2 lambda)^N) / 2
Q lambdaInitial = Alg.R(11, 20);
var attendusRecursion = new Q[] { Alg.R(11, 20), Alg.R(99, 200), Alg.R(1001, 2000), Alg.R(9999, 20000), Alg.R(100001, 200000), Alg.R(999999, 2000000), Alg.R(10000001, 20000000) };
Console.WriteLine("lambda_1 = " + FmtQ(lambdaInitial, 2) + " (dans le trou : NPT mais pas 1-distillable en d=3)");
Console.WriteLine("N     lambda_N");
Q puissanceRecursion = 1;
Q maximumRecursion = 0;
bool sousSeuilRecursion = true;
for (int n = 1; n <= 7; n++)
{
    puissanceRecursion *= (Q)1 - (Q)2 * lambdaInitial;
    Q lambdaN = ((Q)1 - puissanceRecursion) / (Q)2;
    Alg.Assert(lambdaN == attendusRecursion[n - 1], "Valeur exacte de la récursion");
    if (lambdaN.N * maximumRecursion.D > maximumRecursion.N * lambdaN.D) maximumRecursion = lambdaN;
    sousSeuilRecursion &= lambdaN.N * lambdaInitial.D <= lambdaInitial.N * lambdaN.D;
    Console.WriteLine(n.ToString(CultureInfo.InvariantCulture) + "     " + FmtQ(lambdaN, 6));
}
Alg.Assert(sousSeuilRecursion && maximumRecursion == lambdaInitial, "Borne des copies calculées");
Console.WriteLine("\nPropriete (Duer et al.) sur les N calcules : lambda_N <= lambda_1 = " + FmtQ(lambdaInitial, 6) + " : " + sousSeuilRecursion);
Console.WriteLine("(max observe lambda_N = " + FmtQ(maximumRecursion, 6) + ", les copies oscillent autour de 1/2 sans depasser lambda_1)");

lambda_1 = 0.55 (dans le trou : NPT mais pas 1-distillable en d=3)


N     lambda_N


1     0.550000


2     0.495000


3     0.500500


4     0.499950


5     0.500005


6     0.500000


7     0.500000



Propriete (Duer et al.) sur les N calcules : lambda_N <= lambda_1 = 0.550000 : True


(max observe lambda_N = 0.550000, les copies oscillent autour de 1/2 sans depasser lambda_1)


### Lecture du résultat

Parti de $\lambda_1 = 0.55$ — à peine au-dessus du seuil de séparabilité — le poids antisymétrique des copies successives oscille en s'amortissant autour de $1/2$ (0.55, 0.495, 0.5005, …), conformément à la propriété $\lambda_N \leq \lambda_1$ démontrée par Dür *et al.* : **prendre des copies n'aggrave pas l'intrication détectable**, il l'érode au contraire. Les protocoles de distillation ne peuvent donc pas compter sur un effet de cumul simple, et c'est précisément pourquoi la question « tout état NPT est-il distillable ? » (question ouverte n° 2 de la revue 2009, déjà restreinte aux états de Werner par Horodecki–Horodecki 1999) reste ouverte : les études numériques à 2 et 3 copies (Dür *et al.*, DiVincenzo *et al.*, 2000) n'ont pas fermé le trou.

Par contraste, le volet **PPT** de la question est réglé : PPT $\Rightarrow$ non distillable, car $\rho^{\Gamma} \geq 0$ entraîne $\langle\psi|(\rho^{\otimes n})^{\Gamma}|\psi\rangle \geq 0$ pour tout $|\psi\rangle$ et tout $n$ — les états à enchevêtrement lié (*bound entangled*) sont intriqués et définitivement non distillables.

## 6. Canaux « entanglement-breaking » : quand un canal détruit toute intrication

Un canal quantique est une application $\Phi$ complètement positive et préservant la trace. Horodecki, Shor et Ruskai (2003) définissent :

> **Définition 1.** $\Phi$ est *entanglement-breaking* (EB) si $(I \otimes \Phi)(\Gamma)$ est séparable pour tout état $\Gamma$ — tout état intriqué partagé avec un complice est détruit.

Leurs caractérisations calculables :

- **forme de Holevo** (mesure-puis-prépare) : $\Phi(\rho) = \sum_k R_k \operatorname{tr}(F_k \rho)$ avec $\{F_k\}$ une POVM et $R_k$ des états — le canal ne transmet que du classique ;
- **critère de Choi** : $\Phi$ est EB si et seulement si sa matrice de Choi $C_\Phi = (\Phi \otimes I)(|\Psi\rangle\langle\Psi|)$ (avec $|\Psi\rangle = \frac{1}{\sqrt d}\sum_i |ii\rangle$) est **séparable** ;
- structurellement, un canal d'un qubit est EB dès que l'image de la sphère de Bloch s'aplatit en un plan ou une droite.

L'**indice EB** (Lami–Giovannetti 2015) raffine : le $n$-indice d'un canal est le plus petit nombre d'applications **en série** qui rend le canal composé EB — les canaux EB ont indice 1, les canaux unitaires indice $\infty$. C'est le cadre de la question ouverte n° 38 de la revue 2009 (l'indice « PPT² » et ses variantes).

In [10]:
CQ[,] ChoiDepolarisant(Q p, int d)
{
    var c = new CQ[d * d, d * d];
    for (int a = 0; a < d; a++) for (int b = 0; b < d; b++)
    {
        c[a * d + a, b * d + b] += (CQ)(((Q)1 - p) / (Q)d);
        if (a == b)
            for (int i = 0; i < d; i++) c[i * d + a, i * d + a] += (CQ)(p / (Q)(d * d));
    }
    return c;
}
CQ[,] ProjecteurMaxEntrelace(int d)
{
    var c = new CQ[d * d, d * d];
    for (int a = 0; a < d; a++) for (int b = 0; b < d; b++) c[a * d + a, b * d + b] = (CQ)Alg.R(1, d);
    return c;
}
// Seuil EB du canal depolarisant (Lami-Giovannetti 2015, eq. 33) : EB ssi p >= d/(d+1).
// Le critere de Choi le traduit : C est un etat isotrope, separable ssi F = <Psi|C|Psi> <= 1/d.
Console.WriteLine("d   p        F=<Psi|C|Psi>   vp(C^G_B) min   seuils : p >= d/(d+1) et F <= 1/d");
var fidelitesChoiAttendues = new Q[,] {
    { Alg.R(43, 80), Alg.R(1, 2), Alg.R(37, 80) },
    { Alg.R(17, 45), Alg.R(1, 3), Alg.R(13, 45) }
};
var minimumsChoiAttendus = new Q[,] {
    { Alg.R(-3, 80), 0, Alg.R(3, 80) },
    { Alg.R(-1, 45), 0, Alg.R(1, 45) }
};
for (int d = 2; d <= 3; d++)
{
    Q seuil = Alg.R(d, d + 1);
    Q[] points = { seuil - Alg.R(1, 20), seuil, seuil + Alg.R(1, 20) };
    for (int k = 0; k < points.Length; k++)
    {
        Q p = points[k];
        var choi = ChoiDepolarisant(p, d);
        var transposee = Alg.TpB(choi, d, d);
        CQ fidelite = Alg.Trace(Alg.Mul(ProjecteurMaxEntrelace(d), choi));
        Q minimum = transposee[1, 1].Re - transposee[1, d].Re;
        Alg.Assert(Alg.Trace(choi) == (CQ)1, "Trace de Choi");
        Alg.Assert(fidelite == (CQ)fidelitesChoiAttendues[d - 2, k] && minimum == minimumsChoiAttendus[d - 2, k], "Fidélité et spectre de Choi");
        Alg.Assert(minimum == p / (Q)(d * d) - ((Q)1 - p) / (Q)d, "Valeur propre antisymétrique");
        Console.WriteLine(d.ToString(CultureInfo.InvariantCulture) + "   " + FmtQ(p, 4) + "   " + FmtQ(fidelite.Re, 6) + "        " + FmtQ(minimum, 6, true) + "        (" + FmtQ(seuil, 4) + ", " + FmtQ(Alg.R(1, d), 4) + ")");
    }
}

d   p        F=<Psi|C|Psi>   vp(C^G_B) min   seuils : p >= d/(d+1) et F <= 1/d


2   0.6167   0.537500        -0.037500        (0.6667, 0.5000)


2   0.6667   0.500000        +0.000000        (0.6667, 0.5000)


2   0.7167   0.462500        +0.037500        (0.6667, 0.5000)


3   0.7000   0.377778        -0.022222        (0.7500, 0.3333)


3   0.7500   0.333333        +0.000000        (0.7500, 0.3333)


3   0.8000   0.288889        +0.022222        (0.7500, 0.3333)


### Lecture du résultat

La matrice de Choi du canal dépolarisant est un **état isotrope** de fidélité $F = \langle\Psi|C_\Phi|\Psi\rangle = (1-p) + p/d^2$. Aux deux dimensions testées, les deux seuils basculent **au même point** :

- $p < d/(d+1)$ : $F > 1/d$ et $C^{\Gamma_B}$ a une valeur propre négative — la Choi est intriquée, le canal **n'est pas EB** (il laisse passer de l'intrication) ;
- $p \geq d/(d+1)$ : $F \leq 1/d$ et la Choi devient PPT — pour un état isotrope, séparable $\iff$ PPT $\iff F \leq 1/d$ (revue 2009), donc le canal **est EB** : c'est la formule (33) de Lami–Giovannetti, $\Delta_\lambda \in \mathrm{EB} \iff \lambda \leq 1/(d+1)$ avec $\lambda = 1-p$.

On retrouve au passage la cohérence des sections : à $d=2$ le seuil $p \geq 2/3$ est exactement la frontière où la Choi — un état de la famille du §3 — cesse d'être NPT. Le canal dépolarisant **est** une porte sur le laboratoire Werner : bruit croissant = $p$ croissant = Choi migrant du régime intriqué vers le régime séparable.

In [11]:
// Indice EB du canal depolarisant (Lami-Giovannetti 2015, exemple 2) :
// D_p^n est EB si et seulement si (1-p)^n <= 1/(d+1).
Console.WriteLine("d   p      indice EB par puissances exactes");
var pointsIndice = new Q[] { Alg.R(1, 2), Alg.R(3, 5), Alg.R(13, 20), Alg.R(9, 10) };
var indicesAttendus = new int[] { 2, 2, 2, 1 };
for (int d = 2; d <= 3; d++)
{
    Q seuil = Alg.R(1, d + 1);
    for (int k = 0; k < pointsIndice.Length; k++)
    {
        Q p = pointsIndice[k];
        Q reste = (Q)1 - p;
        Q puissance = 1;
        int n = 0;
        do { puissance *= reste; n++; }
        while (puissance.N * seuil.D > seuil.N * puissance.D);
        Alg.Assert(n == indicesAttendus[k], "Indice EB recalculé");
        Q precedente = puissance / reste;
        Alg.Assert(puissance.N * seuil.D <= seuil.N * puissance.D && (n == 1 || precedente.N * seuil.D > seuil.N * precedente.D), "Minimalité de l'indice EB");
        bool pasEb = p.N * (d + 1) < (BigInteger)d * p.D;
        string marque = pasEb ? "  <-- pas EB (p < d/(d+1))" : "";
        Console.WriteLine(d.ToString(CultureInfo.InvariantCulture) + "   " + FmtQ(p, 2) + "   n = " + n.ToString(CultureInfo.InvariantCulture) + marque);
    }
}

d   p      indice EB par puissances exactes


2   0.50   n = 2  <-- pas EB (p < d/(d+1))


2   0.60   n = 2  <-- pas EB (p < d/(d+1))


2   0.65   n = 2  <-- pas EB (p < d/(d+1))


2   0.90   n = 1


3   0.50   n = 2  <-- pas EB (p < d/(d+1))


3   0.60   n = 2  <-- pas EB (p < d/(d+1))


3   0.65   n = 2  <-- pas EB (p < d/(d+1))


3   0.90   n = 1


### Lecture du résultat

Les indices sont recalculés par puissances rationnelles exactes, avec vérification du seuil et de la minimalité. Pour $0<\lambda=1-p<1$, cette condition équivaut à la formule $n(\Delta_\lambda) = \lceil \log(d+1) / \log(1/\lambda) \rceil$ (éq. 34 de Lami–Giovannetti), sans constituer ici une seconde évaluation numérique indépendante de cette formule. Lecture physique :

- un canal déjà EB ($p \geq d/(d+1)$) a $n = 1$ — une seule application détruit toute intrication ;
- en dessous, chaque application multiplie le paramètre de bruit résiduel $\lambda = 1-p$ par lui-même : l'indice compte combien de passages rapprochent $\lambda^n$ de $1/(d+1)$ — le canal est d'autant plus « dangereux » pour l'intrication que son indice est petit ;
- le canal identité ($p = 0$) aurait $n = \infty$, comme toute transformation unitaire.

L'indice EB et son cousin PPT (« $\Phi^2$ est-il EB dès que $\Phi$ est PPT ? ») donnent leur langage à la question ouverte n° 38 de la revue 2009 — le papier de Lami–Giovannetti cité en référence en calcule les valeurs explicites pour les grandes familles de canaux, dépolarisant en tête.

## 7. Exercices

Trois exercices pour s'approprier les organes du carnet (`WernerD`, la purification du §1, `ChoiDepolarisant`). Les stubs s'exécutent sans erreur ; à vous de les compléter.

### Exercice 1 — Werner en dimension 4

Le §3 a vérifié le seuil $\lambda = 1/2$ en $d = 3$. Refaites la vérification en $d = 4$ : pour $\lambda \in \{0.4, 0.5, 0.6\}$, calculez la valeur propre minimale de $W(\lambda)^{\Gamma_B}$ et constatez que la frontière PPT reste $\lambda = 1/2$ — la revue 2009 affirme que c'est le cas en **toute** dimension.

In [12]:
// Exercice a completer
// Etape 1 : construire W(lambda) pour d = 4 avec WernerD (section 3)
// Etape 2 : pour lambda dans [0.4, 0.5, 0.6], calculer la valeur propre minimale de la transposée partielle
// Etape 3 : verifier que la frontiere PPT est encore lambda = 1/2 et conclure
// TODO etudiant : liste des couples (lambda, valeur propre minimale de W^G)
var resultatEx1 = Array.Empty<(Q lambda, Q minimum)>();

### Exercice 2 — purifier un état de Werner

Le §1 a purifié un qubit mixte. Purifiez maintenant $\rho = \text{Werner2Q}(0.5)$ : diagonalisez-le, construisez $|\Psi\rangle = \sum_k \sqrt{w_k}\,|a_k\rangle|k\rangle$ (système 4 dimensions × ancille 4 dimensions) et vérifiez que la trace partielle sur l'ancille redonne $\rho$ exactement. Le rang de Schmidt de votre purification vous surprend-il, au vu du §2 ?

In [13]:
// Exercice a completer
// Etape 1 : diagonaliser rho = Werner2Q(0.5) (4 valeurs propres)
// Etape 2 : construire |Psi> = somme_k sqrt(w_k) |a_k>|k> sur systeme x ancille (4 x 4)
// Etape 3 : verifier tr_ancille(|Psi><Psi|) == rho comme au §1
// Indice : la purification d'un etat de rang 4 est generiquement de rang de Schmidt 4
// TODO etudiant : le ket de purification |Psi>
var resultatEx2 = Array.Empty<CQ>();

### Exercice 3 — seuil EB du canal dépolarisant en dimension 3

Le §6 a localisé le seuil EB par trois points. Faites le balayage propre : pour $p$ de $0.6$ à $0.85$ par pas de $0.05$, calculez la valeur propre minimale de $C_\Phi^{\Gamma_B}$ pour le canal dépolarisant avec $d = 3$, repérez le changement de signe et comparez au seuil théorique $p \geq d/(d+1) = 3/4$.

In [14]:
// Exercice a completer
// Etape 1 : balayer p = 0.60, 0.65, ..., 0.85 avec ChoiDepolarisant(p, 3)
// Etape 2 : pour chaque p, afficher la valeur propre minimale de la transposée partielle
// Etape 3 : localiser le changement de signe et comparer a la frontiere p = 3/4
// TODO etudiant : le seuil numerique observe
var resultatEx3 = Array.Empty<Q>();

## À retenir

1. **Mixte ≠ superposition** : la pureté $\operatorname{tr}(\rho^2)$ sépare statistique classique (§1) et cohérence quantique ; toute matrice densité est l'état réduit d'un état pur plus grand (purification).
2. **Intrication d'un pur = spectre de l'état réduit** (Schmidt, §2) ; pour un mixte, il faut la décomposition en produits — indécidable par le seul spectre, d'où les critères.
3. **Werner $\lambda$** : une famille à un paramètre où séparable $\iff$ PPT $\iff \lambda \leq 1/2$ en toute dimension (§3), laboratoire de tout le reste.
4. **PPT** : $\rho^{\Gamma_B} \geq 0$ est nécessaire pour la séparabilité (Peres), suffisante en $2\times2$ et $2\times3$ (Horodecki *et al.*), définitivement insuffisante au-delà — enchevêtrement lié (§4).
5. **Distillabilité** : le critère de Schmidt rang 2 sépare NPT-distillable du trou $1/2 < \lambda \leq 3(d-1)/(2(2d-1))$ ; PPT $\Rightarrow$ non distillable ; « tout NPT est-il distillable ? » reste ouvert (§5, question ouverte n° 2).
6. **Canaux EB** : le canal casse toute intrication $\iff$ sa Choi est séparable $\iff$ il ne transmet que du classique (forme de Holevo) ; l'indice EB compte les applications en série pour y arriver (§6, question ouverte n° 38).

Le fil directeur : **chaque affirmation a été un nombre affiché par une cellule** — spectres, seuils, frontières numériques confondus avec les formules publiées, jamais seulement recopiées.

## Références

Toutes les publications sont lues et citées **depuis leur copie archivée** au gisement partagé `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\` (convention du dépôt : les PDF sous droits ne sont jamais committés) :

- A. Peres, *Separability Criterion for Density Matrices* (1996, arXiv 9604005) — `1996 - Peres - Separability Criterion for Density Matrices (arXiv 9604005).pdf` : critère PPT, éq. 3 et 7-8, exemples des §4.
- M. Horodecki, P. Horodecki, R. Horodecki, *Separability of Mixed States: Necessary and Sufficient Conditions* (1996, arXiv 9605038) — `1996 - Horodecki et al - Separability of Mixed States - Necessary and Sufficient Conditions (arXiv 9605038).pdf` : PPT décisif en $2\times2$ et $2\times3$.
- C. H. Bennett, G. Brassard, S. Popescu *et al.*, *Purification of Noisy Entanglement and Faithful Teleportation via Noisy Channels* (1996, arXiv 9511027) — `1996 - Bennett et al - Purification of Noisy Entanglement and Faithful Teleportation via Noisy Channels (arXiv 9511027).pdf` : la distillation.
- W. Dür, J. I. Cirac, M. Lewenstein, D. Bruß, *Distillability and Partial Transposition in Bipartite Systems* (2000, arXiv 9910022) — `2000 - Duer Cirac Lewenstein Bruss - Distillability and Partial Transposition in Bipartite Systems (arXiv 9910022).pdf` : seuil $3(d-1)/(2(2d-1))$, récursion $\lambda_N$ (§5).
- D. P. DiVincenzo, P. W. Shor, J. A. Smolin, B. M. Terhal, A. V. Thapliyal, *Evidence for Bound Entangled States with Negative Partial Transpose* (2000, arXiv 9910026) — `2000 - DiVincenzo et al - Evidence for Bound Entangled States with Negative Partial Transpose (arXiv 9910026).pdf` : le cadre numérique pseudo-$n$-copies (§5).
- M. Horodecki, P. W. Shor, M. B. Ruskai, *Entanglement Breaking Channels* (2003, arXiv 0302031) — `2003 - Horodecki Shor Ruskai - Entanglement Breaking Channels (arXiv 0302031).pdf` : définition EB, forme de Holevo, critère de Choi (§6).
- R. Horodecki, P. Horodecki, M. Horodecki, K. Horodecki, *Quantum Entanglement* (2009, arXiv 0702225) — `2009 - Horodecki et al - Quantum Entanglement (arXiv 0702225).pdf` : revue — états de Werner (éq. 71), critère de distillabilité (note 49), états isotropes, questions ouvertes n° 2 et 38.
- G. Lami, V. Giovannetti, *Entanglement-Breaking Indices* (2015, arXiv 1411.2517) — `2015 - Lami Giovannetti - Entanglement-Breaking Indices (arXiv 1411.2517).pdf` : seuil $\lambda \leq 1/(d+1)$ (éq. 33) et $n$-indice du dépolarisant (éq. 34, exemple 2).

L'état de Werner original (R. F. Werner, *Phys. Rev. A* **40**, 4277, 1989) n'a pas de préprint archivé au gisement ; il est cité ici via la paramétrisation de la revue 2009 (éq. 71), conformément à la règle de lecture avant attribution.